In [2]:
# ======================================================================
# 00_shared_utils.ipynb
# FinGap-Governance | Shared utilities (self-contained reference module)
# ----------------------------------------------------------------------
# Standalone: running this notebook top-to-bottom defines every path,
# the closed provision vocabulary, the four-link chain / failure taxonomy,
# the case dataclasses, the citation guards, and the result writers used
# by notebooks 01-03. Notebooks 01-03 each embed their own copy of this
# preamble, so none of them depend on this file at run time.
# ======================================================================

from __future__ import annotations
import os, re, json, glob, csv
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

# ---- 1. Project paths -------------------------------------------------
ROOT       = Path(os.environ.get("FINGAP_ROOT", ".")).resolve()
DATA       = ROOT / "data"
ANOMALY    = DATA / "anomaly"    / "raw"
ADVISORY   = DATA / "advisory"   / "labeled"
ACCOUNTING = DATA / "accounting" / "labeled"
CASES      = DATA / "cases"
RESULTS    = ROOT / "results"
FIGURES    = RESULTS / "figures"
TABLES     = RESULTS / "tables"
for _p in (FIGURES, TABLES):
    _p.mkdir(parents=True, exist_ok=True)

# ---- 2. Closed provision vocabulary (graph nodes) ---------------------
# Canonical identifiers only; the GraphRAG condition may cite ONLY these.
# This is what makes hallucination structurally impossible in the stub.
PROVISIONS = {
    # --- Anomaly detection: Act on Reporting and Use of Specific
    #     Financial Transaction Information (TFTIA) ---
    "TFTIA-Art4":      "TFTIA Art. 4 (Suspicious Transaction Report; "
                       "'reasonable grounds to suspect', report to FIU without delay)",
    "TFTIA-Art4-2":    "TFTIA Art. 4-2 (Currency Transaction Report)",
    "TFTIA-Enf-Art8-2":"TFTIA Enforcement Decree Art. 8-2 "
                       "(CTR threshold KRW 10M, 30-day filing window)",
    "TFTIA-Art2-4":    "TFTIA Art. 2(4) (definition of money laundering)",
    # --- Advisory: Financial Consumer Protection Act (FCPA) ---
    "FCPA-Art17":      "FCPA Art. 17 (suitability)",
    "FCPA-Art18":      "FCPA Art. 18 (appropriateness)",
    "FCPA-Art19":      "FCPA Art. 19 (duty to explain)",
    "FCPA-Art20":      "FCPA Art. 20 (prohibition of unfair business conduct)",
    "FCPA-Art21":      "FCPA Art. 21 (prohibition of improper solicitation)",
    "FCPA-Art22":      "FCPA Art. 22 (advertising regulation)",
    # --- Accounting: K-IFRS 1115 / IFRS 15 ---
    "KIFRS1115-P35":   "K-IFRS 1115 para. 35 (performance obligation "
                       "satisfied over time)",
    "KIFRS1115-P35-3": "K-IFRS 1115 para. 35(3) (no alternative use + "
                       "enforceable right to payment)",
    "KIFRS1018":       "K-IFRS 1018 (superseded by 1115) — citation-blocked",
    "KIFRS1011":       "K-IFRS 1011 (superseded by 1115) — citation-blocked",
}
SUPERSEDED = {"KIFRS1018", "KIFRS1011"}   # exist only to be refused

# ---- 3. Four-link chain + failure taxonomy ----------------------------
CHAIN = ["observation", "subsumption", "determination", "justification"]
FAILURES = {
    "boundary_misjudgment": "signature: normative boundary decided but "
                            "boundary ground not cited",
    "provision_omission":   "determination made, no provision cited",
    "misattribution":       "wrong / superseded provision cited",
    "logic_disconnection":  "correct provision, not linked to facts",
    "hallucination":        "non-existent provision/paragraph generated",
}

# ---- 4. Data structures for a single walkthrough case -----------------
@dataclass
class Verdict:
    condition: str                       # "direct" | "vanilla_rag" | "graph_rag"
    determination: str                   # the normative conclusion produced
    cited: list = field(default_factory=list)   # provision keys claimed
    traceable: bool = False              # justification linked to facts?
    failure: Optional[str] = None        # key of FAILURES, if any

@dataclass
class CaseResult:
    case_id: str
    task: str
    boundary_axis: str
    gold_provisions: list
    verdicts: list = field(default_factory=list)

# ---- 5. Citation guards standing in for the three conditions ----------
def check_citations(cited: list) -> dict:
    """Flag hallucinated (out-of-vocabulary) and superseded citations."""
    return {
        "hallucinated": [c for c in cited if c not in PROVISIONS],
        "superseded":   [c for c in cited if c in SUPERSEDED],
    }

def graph_guard(cited: list) -> list:
    """GraphRAG closed-vocabulary guard: keep only real, live nodes."""
    return [c for c in cited if c in PROVISIONS and c not in SUPERSEDED]

# ---- 6. Result writers ------------------------------------------------
def save_table(name: str, header: list, rows: list) -> Path:
    out = TABLES / f"{name}.csv"
    with open(out, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)
    return out

def save_json(name: str, obj) -> Path:
    out = TABLES / f"{name}.json"
    out.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return out

print("shared utilities ready")
print("provision vocabulary size:", len(PROVISIONS),
      "| superseded (blocked):", sorted(SUPERSEDED))

shared utilities ready
provision vocabulary size: 14 | superseded (blocked): ['KIFRS1011', 'KIFRS1018']
